In [0]:
%pip install --upgrade mlflow databricks-sdk dspy openai
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


   
# Model Migration: It's Not as Simple as Swapping Prompts

Below is an example that updates a prompt for a different model. It's a simple prompt — **classify transaction risk** — so you'll likely see bigger improvements for more complex use cases. We'll go from GPT-5 to Gemma 3/GPT-OSS-20B.

This notebook is designed to classify transaction descriptions into risk categories for Plaid's fraud detection and anti-money laundering (AML) systems.

Make sure you have access to Databricks Foundation Model APIs to run this successfully.

In [0]:
import mlflow
import openai
from mlflow.genai.optimize import GepaPromptOptimizer
from mlflow.genai.scorers import Correctness
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Change the catalog and schema to your catalog and schema
catalog = "main"
schema = "default"
prompt_registry_name = "plaid_transaction_risk_classifier"
prompt_location = f"{catalog}.{schema}.{prompt_registry_name}"

openai_client = w.serving_endpoints.get_open_ai_client()

# Register the initial prompt
prompt = mlflow.genai.register_prompt(
    name=prompt_location,
    template="Classify the transaction risk level of the following operation: {{description}}",
)


# Define the prediction function
def predict_fn(description: str) -> str:
    prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_location}/1")
    completion = openai_client.chat.completions.create(
        model="databricks-gpt-5",
        # Load prompt template using PromptVersion.format()
        messages=[{"role": "user", "content": prompt.format(description=description)}],
    )
    return completion.choices[0].message.content

/home/spark-5ce191d6-fb3a-459f-8f87-6e/.ipykernel/3700/command-2914629301419267-4246092180:15: DeprecationWarning: get_open_ai_client() is deprecated. Please install the databricks-openai package and use 'from databricks_openai import DatabricksOpenAI' instead. See https://api-docs.databricks.com/python/databricks-ai-bridge/latest/databricks_openai.html for more information.
  openai_client = w.serving_endpoints.get_open_ai_client()


   
# Test Your Function

Notice how accurately the model can classify the input with a basic prompt. Although accurate, it's not aligned to any specific task or use case that we're targeting for Plaid's risk detection pipeline.

In [0]:
from IPython.display import Markdown

# Example of a suspicious financial transaction
output = predict_fn("Bank account showing rapid succession of large wire transfers to offshore accounts within 24 hours of account linking, with amounts just below the $10,000 CTR threshold and no prior transaction history.")

Markdown(output)

High risk.

Rationale:
- Rapid succession of large wire transfers shortly after account opening/linking is a classic red flag for layering in money laundering.
- Transfers to offshore accounts increase jurisdictional and transparency risk.
- Amounts structured just below the $10,000 CTR threshold indicate potential structuring to evade reporting.
- No prior transaction history or established activity pattern heightens anomaly and KYC mismatch risk.

Recommended actions:
- File a Suspicious Activity Report (SAR) per regulatory obligations.
- Place temporary holds or enhanced monitoring on further transfers.
- Conduct enhanced due diligence (verify source of funds, purpose of transfers, beneficiary details).
- Review for possible structuring across related accounts.

   
# Optimization Against Data

Now we'll provide some data with expected answers and facts. This will help optimize our model to behave and produce results that align with Plaid's risk detection use cases.

In this case, we want the model to produce a single word from a selection of five risk categories. It should only produce that word without any additional explanation.

**Risk Categories:**
- HIGH_RISK: Transactions with clear indicators of fraud or money laundering
- MEDIUM_RISK: Transactions requiring detailed review
- LOW_RISK: Transactions with minor anomalies
- NO_RISK: Normal financial transactions
- REQUIRES_INVESTIGATION: Cases needing specialized audit

In [0]:
# Training data with inputs and expected outputs
dataset = [
    {
        "inputs": {"description": "Bank account showing rapid succession of large wire transfers to offshore accounts within 24 hours of account linking, with amounts just below the $10,000 CTR threshold and no prior transaction history."},
        "expectations": {"expected_facts": ["The classification must be 'HIGH_RISK', 'MEDIUM_RISK', 'LOW_RISK', 'NO_RISK', 'REQUIRES_INVESTIGATION'"]}
    },
    {
        "inputs": {"description": "Bank account showing rapid succession of large wire transfers to offshore accounts within 24 hours of account linking, with amounts just below the $10,000 CTR threshold and no prior transaction history."},
        "outputs": {"response": "HIGH_RISK"},
        "expectations": {"expected_response": "HIGH_RISK"}
    },
    {
        "inputs": {"description": "Individual with consistent direct deposit from a verified employer, regular bill payments, and savings transfers maintained over 3 years with no suspicious activity flags."},
        "expectations": {"expected_facts": ["The classification must be 'HIGH_RISK', 'MEDIUM_RISK', 'LOW_RISK', 'NO_RISK', 'REQUIRES_INVESTIGATION'"]}
    },
    {
        "inputs": {"description": "Individual with consistent direct deposit from a verified employer, regular bill payments, and savings transfers maintained over 3 years with no suspicious activity flags."},
        "outputs": {"response": "NO_RISK"},
        "expectations": {"expected_response": "NO_RISK"}
    },
    {
        "inputs": {"description": "Multiple accounts linked from the same device within minutes, each with different identity credentials, attempting to initiate ACH transfers to the same destination account."},
        "expectations": {"expected_facts": ["The classification must be 'HIGH_RISK', 'MEDIUM_RISK', 'LOW_RISK', 'NO_RISK', 'REQUIRES_INVESTIGATION'"]}
    },
    {
        "inputs": {"description": "Multiple accounts linked from the same device within minutes, each with different identity credentials, attempting to initiate ACH transfers to the same destination account."},
        "outputs": {"response": "HIGH_RISK"},
        "expectations": {"expected_response": "HIGH_RISK"}
    },
    {
        "inputs": {"description": "Small business account showing a 20% revenue increase compared to prior quarter due to seasonal holiday sales, with all transactions matching typical merchant category codes."},
        "expectations": {"expected_facts": ["The classification must be 'HIGH_RISK', 'MEDIUM_RISK', 'LOW_RISK', 'NO_RISK', 'REQUIRES_INVESTIGATION'"]}
    },
    {
        "inputs": {"description": "Small business account showing a 20% revenue increase compared to prior quarter due to seasonal holiday sales, with all transactions matching typical merchant category codes."},
        "outputs": {"response": "LOW_RISK"},
        "expectations": {"expected_response": "LOW_RISK"}
    },
    {
        "inputs": {"description": "Account holder's linked bank accounts across multiple institutions show coordinated large withdrawals within a 48-hour window, followed by cryptocurrency exchange deposits and immediate transfers to foreign wallets."},
        "expectations": {"expected_facts": ["The classification must be 'HIGH_RISK', 'MEDIUM_RISK', 'LOW_RISK', 'NO_RISK', 'REQUIRES_INVESTIGATION'"]}
    },
    {
        "inputs": {"description": "Account holder's linked bank accounts across multiple institutions show coordinated large withdrawals within a 48-hour window, followed by cryptocurrency exchange deposits and immediate transfers to foreign wallets."},
        "outputs": {"response": "REQUIRES_INVESTIGATION"},
        "expectations": {"expected_response": "REQUIRES_INVESTIGATION"}
    },
    {
        "inputs": {"description": "New account linked through Plaid showing transaction history with irregular income patterns, several returned ACH payments, but no clear indicators of fraud or identity theft."},
        "expectations": {"expected_facts": ["The classification must be 'HIGH_RISK', 'MEDIUM_RISK', 'LOW_RISK', 'NO_RISK', 'REQUIRES_INVESTIGATION'"]}
    },
    {
        "inputs": {"description": "New account linked through Plaid showing transaction history with irregular income patterns, several returned ACH payments, but no clear indicators of fraud or identity theft."},
        "outputs": {"response": "MEDIUM_RISK"},
        "expectations": {"expected_response": "MEDIUM_RISK"}
    }
]

# Optimize the prompt
result = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=dataset,
    prompt_uris=[prompt.uri],
    optimizer=GepaPromptOptimizer(reflection_model="databricks:/databricks-gpt-5-2"),
    scorers=[Correctness(model="databricks:/databricks-gpt-5")],
)

# Use the optimized prompt
optimized_prompt = result.optimized_prompts[0]
print(f"Optimized template: {optimized_prompt.template}")

2026/03/25 02:56:38 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/03/25 02:56:38 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
2026-03-25 02:56:41,711 3700 ERROR _handle_rpc_error GRPC Error received
Traceback (most recent call last):
  File "/databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/client/core.py", line 1724, in config
    resp = self._stub.Config(req, metadata=self.metadata())
  File "/databricks/python/lib/python3.10/site-packages/grpc/_interceptor.py", line 277, in __call__
    response, ignored_call = self._with_call(
  File "/databricks/python/lib/python3.10/site-packages/grpc/_interceptor.py", line 332, in _with_call
    return call.r

Iteration 0: Base program full valset score: 0.75 over 12 / 12 examples
Iteration 1: Selected program 0 score: 0.75
Iteration 1: Proposed new text for main.default.plaid_transaction_risk_classifier: You are a transaction risk classifier for Plaid-linked bank activity descriptions.

Task:
Given a single free-text description of an operation/behavior pattern, output ONLY one risk label from this fixed set:
- HIGH_RISK
- MEDIUM_RISK
- LOW_RISK
- NO_RISK
- REQUIRES_INVESTIGATION

Output format rules (strict):
1) Return exactly one of the five labels above, in all caps, with underscores.
2) Do not output explanations, rationales, bullet points, or any extra text.

Classification guidance (domain-specific signals):
- HIGH_RISK: Clear, multiple AML/fraud red flags suggesting likely illicit activity (e.g., rapid large wire transfers immediately after account linking; offshore transfers; structuring just below $10,000 CTR threshold; coordinated large withdrawals across multiple institutions in 

[Trace(trace_id=tr-83eae6429b40469417dae73b8ac0b645), Trace(trace_id=tr-d5e6e6bcdea085a26bd37ebeb2d3938e), Trace(trace_id=tr-4c831184717dbde60fb8caec8f580649), Trace(trace_id=tr-0a6ff164d13cf4b44a591738f0c51ddc), Trace(trace_id=tr-e4426dff09c8086135c15b22f13c434f), Trace(trace_id=tr-79930541bcbf8b28b5b15a4832915c44), Trace(trace_id=tr-acb5f6040952d8289685475fc6028813), Trace(trace_id=tr-eec3f3f646db1645d7223471eb640235), Trace(trace_id=tr-90e23aabc5a36ea36be22486323bd814), Trace(trace_id=tr-40317cca5fea85d838b992c5edb00dfc)]

In [0]:
print(f"Initial Score: {result.initial_eval_score}\n") 
print(f"Final Score: {result.final_eval_score}")

Initial Score: 0.75

Final Score: 0.75


   
# Let's Review the Changes

Let's test how GPT-5 performs now with the optimized prompt for transaction risk classification.

In [0]:
import mlflow

mlflow.openai.autolog()

def predict_fn(description: str) -> str:
    prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_location}/2") 
    completion = openai_client.chat.completions.create(
        model = "databricks-gpt-5",
        messages=[{"role": "system", "content": prompt.format(description=description)}]
    )
    return completion.choices[0].message.content

In [0]:
# Test case: Suspicious money laundering operation
output = predict_fn(description="Linked bank account receiving multiple structured cash deposits just below the $10,000 reporting threshold from unrelated individuals, with immediate outbound wire transfers to accounts in high-risk jurisdictions.")

Trace(trace_id=tr-bab46caf10eedb30f0cbc2b0d09e1eee)

In [0]:
# Expected answer: HIGH_RISK
output

'High risk\n\nReasons:\n- Structuring: Multiple cash deposits just below $10,000 indicate potential evasion of CTR requirements.\n- Third-party deposits: Funds from unrelated individuals increase suspicion of money mule activity.\n- Rapid movement of funds: Immediate outbound transfers suggest layering to obscure origin.\n- High-risk destinations: Wires to high-risk jurisdictions elevate AML/CTF concerns.\n\nRecommended actions:\n- File a suspicious activity report (SAR) where applicable.\n- Enhance due diligence and temporarily restrict activity pending review.\n- Verify customer identity, source of funds, and purpose of transactions.\n- Monitor for related accounts/patterns.'

   
# Let's Try with Gemma

How well does it work with this optimized prompt on a smaller model?

In [0]:
from IPython.display import Markdown
prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_location}/2")

Markdown(prompt.template)

Classify the transaction risk level of the following operation: {{description}}

In [0]:
def predict_fn_gemma(description: str) -> str:
    prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_location}/2")
    completion = openai_client.chat.completions.create(
        model="databricks-gemma-3-12b",
        messages=[{"role": "system", "content": prompt.format(description=description)},
            {"role": "user", "content": description}],
    )
    return completion.choices[0].message.content

In [0]:
# Test case with Gemma
output = predict_fn_gemma(description="Linked bank account receiving multiple structured cash deposits just below the $10,000 reporting threshold from unrelated individuals, with immediate outbound wire transfers to accounts in high-risk jurisdictions.")

Trace(trace_id=tr-e4bc7824b72e7220dcdd6a4f1d9789ca)

In [0]:
# Expected answer: HIGH_RISK
output

'Okay, let\'s break down the transaction risk level of this operation. This scenario screams **HIGH RISK**. Here\'s a detailed classification and justification:\n\n**Risk Level: HIGH**\n\n**Justification (Why it\'s High Risk):**\n\nThis transaction combines multiple red flags, significantly elevating the risk profile.  Here\'s a breakdown of each element and how they contribute to the overall high-risk assessment:\n\n*   **Structured Deposits Below $10,000 Threshold:** This is a classic structuring tactic used to evade Currency Transaction Reporting (CTR).  Individuals depositing amounts just below the reporting threshold to avoid triggering scrutiny is a major red flag. This suggests an intent to conceal the source and/or movement of funds.\n*   **Multiple Deposits from Unrelated Individuals:**  The fact that the deposits are coming from *unrelated* individuals further raises suspicion. It suggests a potential layering scheme, where different people are being used to move money. It ob

   
# Incorrect Result, Not Surprising

Although we switched to a smaller model, we can see that it's not as simple as giving it an optimized prompt and expecting the same performance improvements.

We need to re-optimize to get a new prompt for the new model based on the original prompt.

Let's do that below. I'll set up a new function to use the Gemma 3 model.

In [0]:
def predict_fn_gemma(description: str) -> str:
    prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_location}/1")
    completion = openai_client.chat.completions.create(
        model="databricks-gemma-3-12b",
        messages=[{"role": "user", "content": prompt.format(description=description)}],
    )
    return completion.choices[0].message.content

In [0]:
result_gemma_oss = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn_gemma,
    train_data=dataset,
    prompt_uris=[prompt.uri],
    optimizer=GepaPromptOptimizer(reflection_model="databricks:/databricks-gpt-5-2"),
    scorers=[Correctness(model="databricks:/databricks-gpt-5")],
)

# Use the optimized prompt
optimized_prompt = result.optimized_prompts[0]
print(f"Optimized template: {optimized_prompt.template}")

2026/03/24 23:54:37 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-2fb3e29a-d7fa-4f56-bb8b-074256dfc677/lib/python3.10/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


Iteration 0: Base program full valset score: 0.3333333333333333 over 12 / 12 examples
Iteration 1: Selected program 0 score: 0.3333333333333333
Iteration 1: Proposed new text for main.default.plaid_transaction_risk_classifier: You are a transaction risk labeler for Plaid-linked banking activity descriptions.

TASK
Given a single natural-language description of an operation/transaction pattern, output exactly ONE risk classification label from the allowed set:

- HIGH_RISK
- MEDIUM_RISK
- LOW_RISK
- NO_RISK
- REQUIRES_INVESTIGATION

OUTPUT FORMAT (STRICT)
- Respond with only the label (all caps with underscore), and nothing else.
- Do not include explanations, bullet points, disclaimers, questions, or extra text.

LABELING GUIDELINES
Use the description’s risk signals to choose the best label:

1) HIGH_RISK
Use when the description contains strong, specific red flags consistent with fraud/AML typologies, such as:
- Structuring/smurfing: repeated transactions just under the $10,000 CTR t

In [0]:
prompt_location

'main.default.plaid_transaction_risk_classifier'

   
# We Should Have Decent Scores Now

In [0]:
print(f"Initial Score: {result_gemma_oss.initial_eval_score}\n") 
print(f"Final Score: {result_gemma_oss.final_eval_score}")

Initial Score: 0.3333333333333333

Final Score: 0.5833333333333334


In [0]:
from IPython.display import Markdown
prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_location}/3")

Markdown(prompt.template)

You are a transaction risk classifier. Given a short description of an operation/behavior, output a SINGLE risk label that best matches the scenario.

Allowed labels (must use EXACTLY one of these, spelled exactly as shown):
- HIGH_RISK
- MEDIUM_RISK
- LOW_RISK
- NO_RISK
- REQUIRES_INVESTIGATION

Task goal:
- Classify the riskiness of the described transaction/operation based on fraud/AML red flags and behavioral anomalies (e.g., velocity, structuring, destination concentration, identity/device mismatch, unusual timing after account linking, offshore transfers, lack of history, atypical patterns).
- Choose the closest single label from the allowed set.

Decision guidance:
- HIGH_RISK: Strong, multiple fraud/AML indicators (e.g., rapid large wires, offshore beneficiaries, structuring just below reporting thresholds, many accounts from same device, different identities, funneling to same destination, immediate high activity after linking). Clear likelihood of fraud/AML abuse.
- MEDIUM_RISK: Some suspicious indicators or moderate anomalies, but not conclusively fraudulent; could be legitimate with explanation.
- LOW_RISK: Largely consistent with expected/normal behavior; only minor explainable anomalies (e.g., seasonal sales increase with typical MCCs).
- NO_RISK: Clearly benign/normal with no meaningful red flags.
- REQUIRES_INVESTIGATION: Information is insufficient or ambiguous such that you cannot responsibly choose HIGH/MEDIUM/LOW/NO; use when key details are missing and the risk cannot be determined.

Output format rules (critical):
- Output ONLY the label (one of the allowed labels) on a single line.
- Do NOT add explanations, extra text, multiple labels, ranges (e.g., “low to moderate”), or new labels (e.g., “CRITICAL”).
- Do NOT ask follow-up questions in the output.

   
# Let's Verify Again

In [0]:
def predict_fn_gemma_updated(description: str) -> str:
    prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_location}/3")
    completion = openai_client.chat.completions.create(
        model="databricks-gemma-3-12b",
        messages=[{"role": "system", "content": prompt.format(description=description)},
            {"role": "user", "content": description}],
    )
    return completion.choices[0].message.content

In [0]:
# New test case: Suspicious international transfers
output = predict_fn_gemma_updated(description="Account linked via Plaid showing recurring outbound transfers to fintech platforms in non-cooperative jurisdictions, with no corresponding inbound salary or business income to justify the transaction volume.")

Trace(trace_id=tr-59316d3ea93fc65ae3781bfbdff25e17)

In [0]:
# Expected answer: REQUIRES_INVESTIGATION
output

'REQUIRES_INVESTIGATION'

# Is It Correct Now?

We can't assume that the existing prompt will work for all models or be efficient for all models. It only takes a few minutes to re-optimize the model!

Let's go back to our experiment to add some aliases.

Now we can use the MLflow prompt registry to load the correct prompts depending on which model we want to use for Plaid's transaction risk classification.

In [0]:
# Gemma 3 12B - Transaction Risk Classifier

def predict_fn(description: str) -> str:
    prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_location}@gemma_3_12b")
    completion = openai_client.chat.completions.create(
        model="databricks-gemma-3-12b",
        messages=[{"role": "system", "content": prompt.format(description=description)},
            {"role": "user", "content": description}],
    )
    return completion.choices[0].message.content

# Test case: Refund fraud pattern
output = predict_fn(description="Established fintech company with 10 years of operation showing consistent transaction volumes, clean compliance audits, and full adherence to BSA/AML reporting requirements across all linked accounts.")
output

'NO_RISK'

Trace(trace_id=tr-bec0d7f85174c1df258883bc08e0843e)

In [0]:
# GPT5 - Transaction Risk Classifier

def predict_fn(description: str) -> str:
    prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_location}@gpt5")
    completion = openai_client.chat.completions.create(
        model = "databricks-gpt-5",
        messages=[{"role": "system", "content": prompt.format(description=description)}]
    )
    return completion.choices[0].message.content

# Test case: Legitimate operation
output = predict_fn(description="Established fintech company with 10 years of operation showing consistent transaction volumes, clean compliance audits, and full adherence to BSA/AML reporting requirements across all linked accounts.")
output

'Risk Level: Low\n\nRationale:\n- Long operating history (10 years) with stable transaction volumes indicates predictable behavior.\n- Clean compliance audits suggest strong internal controls and governance.\n- Demonstrated adherence to BSA/AML reporting across all linked accounts reduces regulatory and financial crime risk.\n- No indicators of adverse events, rapid unexplained growth, or compliance gaps.'

Trace(trace_id=tr-bd1a1ddd01a071546d0c3d35d64ccf0e)